## Batch Inference — Score All Customers Using Registered Model
Loads the registered Random Forest model (olist_churn_model v1) from MLflow 
Model Registry and scores all 93,357 customers. Churn probability scores are 
written back to the Gold layer as a Delta table for Power BI consumption.

Pipeline:
Silver RFM features → Rich feature engineering → Load registered model → 
Score all customers → Write churn scores to Gold layer

In [0]:
import mlflow
import mlflow.sklearn

BRONZE_PATH = "/Volumes/workspace/default/raw_data/bronze"
SILVER_PATH = "/Volumes/workspace/default/raw_data/silver"
GOLD_PATH   = "/Volumes/workspace/default/raw_data/gold"

username = spark.sql("SELECT current_user()").collect()[0][0]


In [0]:
# Load registered model from MLflow Model Registry
model_uri = "models:/workspace.default.olist_churn_model/1"
model = mlflow.sklearn.load_model(model_uri)

print("=== MODEL LOADED FROM REGISTRY ===")
print(f"Model URI     : {model_uri}")
print(f"Model type    : {type(model).__name__}")
print(f"N estimators  : {model.n_estimators}")
print(f"Max depth     : {model.max_depth}")
print("Model ready for batch scoring")

Rebuild Feature Table

In [0]:
from pyspark.sql.functions import (col, to_timestamp, datediff, lit,
    count, sum as spark_sum, avg, max as spark_max,
    min as spark_min, stddev, when)
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Loading all required tables
orders_b      = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
payments_b    = spark.read.format("delta").load(f"{BRONZE_PATH}/payments")
customers_raw = spark.read.format("delta").load(f"{BRONZE_PATH}/customers")
customer_rfm  = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/customer_rfm_features")

print("=== TABLES LOADED ===")
print(f"Orders        : {orders_b.count()}")
print(f"Payments      : {payments_b.count()}")
print(f"Customers     : {customers_raw.count()}")
print(f"Silver RFM    : {customer_rfm.count()}")



In [0]:
# Reference date
max_date = orders_b.agg(
    spark_max(to_timestamp("order_purchase_timestamp"))
).collect()[0][0]
print(f"Reference date: {max_date}")

# ID mapping
id_mapping = customers_raw.select("customer_id", "customer_unique_id")

# Aggregate payments per order
pay_agg = payments_b.groupBy("order_id").agg(
    spark_sum("payment_value").alias("order_value"),
    avg("payment_installments").alias("installments")
)


In [0]:

# Enrich orders
orders_enriched = orders_b \
    .filter(col("order_status") == "delivered") \
    .withColumn("purchase_ts",
                to_timestamp("order_purchase_timestamp")) \
    .withColumn("delivery_ts",
                to_timestamp("order_delivered_customer_date")) \
    .withColumn("estimated_ts",
                to_timestamp("order_estimated_delivery_date")) \
    .withColumn("delivery_delay_days",
                datediff(col("delivery_ts"), col("estimated_ts"))) \
    .join(pay_agg, "order_id", "left") \
    .join(id_mapping, "customer_id", "left")

# Aggregate at customer_unique_id level
rich_features = orders_enriched.groupBy("customer_unique_id").agg(
    count("order_id").alias("frequency"),
    spark_sum("order_value").alias("monetary_value"),
    avg("order_value").alias("avg_order_value"),
    stddev("order_value").alias("stddev_order_value"),
    avg("installments").alias("avg_installments"),
    avg("delivery_delay_days").alias("avg_delivery_delay"),
    spark_max("delivery_delay_days").alias("max_delivery_delay"),
    datediff(lit(max_date),
             spark_max("purchase_ts")).alias("recency_days"),
    datediff(spark_max("purchase_ts"),
             spark_min("purchase_ts")).alias("customer_lifespan_days")
)



In [0]:
# Joining Silver labels for state and freight
silver_labels = customer_rfm.select(
    "customer_unique_id", "is_churned",
    "customer_state", "avg_freight_value"
)

rich_df = rich_features \
    .join(silver_labels, "customer_unique_id", "left") \
    .dropna(subset=["is_churned"])

# Fill nulls
rich_df_clean = rich_df \
    .withColumn("stddev_order_value",
        when(col("stddev_order_value").isNull(), 0)
        .otherwise(col("stddev_order_value"))) \
    .withColumn("avg_delivery_delay",
        when(col("avg_delivery_delay").isNull(), 0)
        .otherwise(col("avg_delivery_delay"))) \
    .withColumn("max_delivery_delay",
        when(col("max_delivery_delay").isNull(), 0)
        .otherwise(col("max_delivery_delay")))

print("")
print(f"=== RICH FEATURE TABLE ===")
print(f"Rows    : {rich_df_clean.count()}")
print(f"Columns : {rich_df_clean.columns}")

Prepare Features + Run Inference

In [0]:
# Convert to Pandas
feature_cols = [
    "frequency", "monetary_value", "avg_order_value",
    "stddev_order_value", "avg_installments",
    "avg_delivery_delay", "max_delivery_delay",
    "customer_lifespan_days", "avg_freight_value",
    "customer_state"
]

df_inference = rich_df_clean.select(
    ["customer_unique_id"] + feature_cols + ["is_churned"]
).dropna().toPandas()

print(f"=== INFERENCE DATASET ===")
print(f"Total customers to score : {df_inference.shape[0]}")

# Encode state
le = LabelEncoder()
df_inference["customer_state_encoded"] = le.fit_transform(
    df_inference["customer_state"]
)
df_inference = df_inference.drop(columns=["customer_state"])


In [0]:
# Feature matrix
X_inference = df_inference[[
    "frequency", "monetary_value", "avg_order_value",
    "stddev_order_value", "avg_installments",
    "avg_delivery_delay", "max_delivery_delay",
    "customer_lifespan_days", "avg_freight_value",
    "customer_state_encoded"
]]

# Run batch inference
df_inference["churn_probability"]  = model.predict_proba(X_inference)[:,1]
df_inference["churn_prediction"]   = model.predict(X_inference)
df_inference["churn_risk_segment"] = df_inference["churn_probability"].apply(
    lambda p: "High Risk"   if p >= 0.7
    else "Medium Risk" if p >= 0.4
    else "Low Risk"
)

print("")
print("=== INFERENCE COMPLETE ===")
print(f"Customers scored         : {df_inference.shape[0]}")
print(f"Predicted churned        : {df_inference['churn_prediction'].sum()}")
print(f"Predicted active         : {(df_inference['churn_prediction']==0).sum()}")
print("")
print("=== CHURN RISK SEGMENT DISTRIBUTION ===")
print(df_inference["churn_risk_segment"].value_counts().to_string())
print("")
print("=== SAMPLE SCORES (top 5 highest churn probability) ===")
print(df_inference[["customer_unique_id",
                     "churn_probability",
                     "churn_risk_segment",
                     "is_churned"]] \
    .sort_values("churn_probability", ascending=False) \
    .head(5).to_string())

Write Scores to Gold Layer

In [0]:
from pyspark.sql.functions import current_timestamp

# Convert back to Spark DataFrame
churn_scores_spark = spark.createDataFrame(
    df_inference[[
        "customer_unique_id",
        "churn_probability",
        "churn_prediction",
        "churn_risk_segment",
        "is_churned"
    ]]
).withColumn("scored_at", current_timestamp())

print("=== CHURN SCORES TABLE PREVIEW ===")
churn_scores_spark.show(5)

print(f"Rows to write : {churn_scores_spark.count()}")

# Write to Gold as Delta
churn_scores_spark.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/customer_churn_scores")

print("")
print("=== GOLD TABLE WRITTEN ===")
print(f"Table    : customer_churn_scores")
print(f"Location : {GOLD_PATH}/customer_churn_scores")
print(f"Rows     : {churn_scores_spark.count()}")
print("Batch inference complete — Gold layer updated")

In [0]:
# Verifying Gold table written correctly
gold_scores = spark.read.format("delta") \
    .load(f"{GOLD_PATH}/customer_churn_scores")

print("=== GOLD TABLE VERIFIED ===")
print(f"Total rows         : {gold_scores.count()}")
print("")

print("=== RISK SEGMENT DISTRIBUTION IN GOLD ===")
gold_scores.groupBy("churn_risk_segment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

print("=== AVG CHURN PROBABILITY BY SEGMENT ===")
gold_scores.groupBy("churn_risk_segment") \
    .agg(avg("churn_probability").alias("avg_churn_prob")) \
    .orderBy("avg_churn_prob", ascending=False) \
    .show()

print("All Gold tables ready for Power BI export")
print("Gold tables available:")
print(f"  - {GOLD_PATH}/customer_churn_segments")
print(f"  - {GOLD_PATH}/revenue_by_state")
print(f"  - {GOLD_PATH}/monthly_revenue_trend")
print(f"  - {GOLD_PATH}/customer_churn_scores")